# Practical 13 - Security, Monitoring, Snapshots and Failure Analysis

This notebook runs against a dedicated secured Elasticsearch node on
`http://127.0.0.1:9201`. The node is started from the included
`docker-compose.yaml` with `xpack.security.enabled=true`.

**Tasks**

1. Configure a secured Elasticsearch environment.
2. Create a user.
3. Create a role.
4. Restrict index access.
5. Generate an API key.
6. Inspect cluster health.
7. Inspect node statistics.
8. Investigate JVM memory usage.
9. Configure slow logs.
10. Create a snapshot repository.
11. Create a snapshot.
12. Restore a snapshot.
13. Investigate an unassigned shard.
14. Investigate an indexing failure.

The compose stack uses a separate data volume and port `9201`, so the
unsecured Practical 9-12 cluster on port `9200` is not modified.

## Setup and secured environment

Security is enabled in `docker-compose.yaml` and
`config/elasticsearch.yml`:

```yaml
xpack.security.enabled: "true"
xpack.security.http.ssl.enabled: "false"
xpack.security.transport.ssl.enabled: "false"
```

HTTP TLS is deliberately disabled for this local lab so the examples stay
readable. Authentication and authorization are still enforced: every
administrative request uses the `elastic` superuser password, and a separate
analyst user is tested against restricted indices.

Start the stack before running the notebook:

```powershell
docker compose up -d
```

The notebook must be executed from the `Practical_13` directory.

In [1]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

from elasticsearch import (
    AuthenticationException,
    AuthorizationException,
    BadRequestError,
    Elasticsearch,
    NotFoundError,
    helpers,
)

from generate_data import build_private_events, build_public_events

HERE = Path.cwd()
assert HERE.name == "Practical_13", f"Run this notebook from Practical_13, not {HERE}"

ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

ES_URL = "http://127.0.0.1:9201"
ELASTIC_PASSWORD = os.environ.get("P13_ELASTIC_PASSWORD", "P13_elastic!123")
CONTAINER = "p13-elasticsearch"

USERNAME = "p13_analyst"
USER_PASSWORD = os.environ.get("P13_USER_PASSWORD", "P13_user!123")
ROLE = "p13_public_reader"

PUBLIC_INDEX = "p13-public-events"
PRIVATE_INDEX = "p13-private-events"
SLOWLOG_INDEX = "p13-slowlog-demo"
FAILURE_INDEX = "p13-failure-demo"
UNASSIGNED_INDEX = "p13-unassigned-demo"

SNAPSHOT_REPO = "p13-snapshot-repo"
SNAPSHOT_NAME = "p13-public-snapshot"
RESTORED_INDEX = "p13-public-events-restored"

PUBLIC_TOTAL = 1_000
PRIVATE_TOTAL = 500

MAPPING = json.loads((HERE / "mapping.json").read_text(encoding="utf-8"))
EVENT_MAPPING = MAPPING["events"]
FAILURE_MAPPING = MAPPING["failure_demo"]

SLOWLOG_SETTINGS = {
    "index.search.slowlog.threshold.query.warn": "0ms",
    "index.search.slowlog.threshold.query.info": "0ms",
    "index.search.slowlog.threshold.fetch.warn": "0ms",
    "index.search.slowlog.threshold.fetch.info": "0ms",
    "index.indexing.slowlog.threshold.index.warn": "0ms",
    "index.indexing.slowlog.threshold.index.info": "0ms",
}

es = Elasticsearch(
    ES_URL,
    basic_auth=("elastic", ELASTIC_PASSWORD),
    request_timeout=120,
)
info = plain = None


def plain(value):
    if hasattr(value, "body") and not isinstance(value, (dict, list)):
        return plain(value.body)
    if hasattr(value, "items"):
        return {key: plain(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(item) for item in value]
    return value


def delete_index(name):
    if es.indices.exists(index=name):
        es.indices.delete(index=name)


def bulk_load(index, records, id_field, chunk_size=500):
    started = time.perf_counter()
    loaded, errors = helpers.bulk(
        es,
        (
            {"_index": index, "_id": record[id_field], "_source": record}
            for record in records
        ),
        chunk_size=chunk_size,
        raise_on_error=False,
    )
    es.indices.refresh(index=index)
    seconds = time.perf_counter() - started
    return {
        "documents": int(loaded),
        "errors": len(errors) if isinstance(errors, list) else int(errors),
        "seconds": round(seconds, 3),
        "docs_per_second": round(loaded / seconds, 1) if seconds else 0.0,
    }


def docker_slowlog_entries(minutes=5):
    if shutil.which("docker") is None:
        return [], "docker CLI not found"
    completed = subprocess.run(
        ["docker", "logs", "--since", f"{minutes}m", CONTAINER],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=60,
    )
    if completed.returncode != 0:
        return [], completed.stderr.strip()[-500:]
    entries = []
    for line in completed.stdout.splitlines():
        if (
            "elasticsearch.index_search_slowlog" not in line
            and "elasticsearch.index_indexing_slowlog" not in line
        ):
            continue
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return entries, None


def slowlog_summary(entry):
    return {
        "level": entry.get("log.level"),
        "logger": entry.get("log.logger"),
        "message": entry.get("elasticsearch.slowlog.message"),
        "took": entry.get("elasticsearch.slowlog.took"),
        "took_millis": entry.get("elasticsearch.slowlog.took_millis"),
        "source": entry.get("elasticsearch.slowlog.source"),
    }


info = plain(es.info())
authenticated = plain(es.security.authenticate())
anonymous_client = Elasticsearch(ES_URL, request_timeout=10)
anonymous_status = None
try:
    anonymous_client.security.authenticate()
except AuthenticationException as exc:
    anonymous_status = exc.status_code

security_environment = {
    "url": ES_URL,
    "cluster_name": info["cluster_name"],
    "node_name": info["name"],
    "version": info["version"]["number"],
    "authenticated_user": authenticated["username"],
    "authenticated_roles": authenticated["roles"],
    "anonymous_status": anonymous_status,
}

print(json.dumps(security_environment, indent=2))
assert info["cluster_name"] == "p13-secured-cluster"
assert authenticated["username"] == "elastic"
assert "superuser" in authenticated["roles"]
assert anonymous_status == 401
print("security is enabled; unauthenticated access was rejected with 401.")

{
  "url": "http://127.0.0.1:9201",
  "cluster_name": "p13-secured-cluster",
  "node_name": "p13-node",
  "version": "9.2.0",
  "authenticated_user": "elastic",
  "authenticated_roles": [
    "superuser"
  ],
  "anonymous_status": 401
}
security is enabled; unauthenticated access was rejected with 401.


## Task 1 - Configure a secured Elasticsearch environment

The secured environment is defined by:

- `docker-compose.yaml`: dedicated node, port `9201`, security enabled, separate volume
- `config/elasticsearch.yml`: cluster name, single-node discovery, snapshot path, security flags

The live authentication check above confirms that the cluster is not accepting
anonymous callers and that the admin client is authenticated as `elastic`.

## Task 2 - Create a user

The analyst user is created without roles first. The role is added in the next
task so the two objects can be examined separately.

In [2]:
try:
    es.security.delete_user(username=USERNAME)
except NotFoundError:
    pass

user_create_response = plain(
    es.security.put_user(
        username=USERNAME,
        password=USER_PASSWORD,
        roles=[],
        full_name="Practical 13 Analyst",
        email="p13-analyst@example.test",
    )
)
created_user = plain(es.security.get_user(username=USERNAME))[USERNAME]

print("create response:", user_create_response)
print("user snapshot:")
print(json.dumps(created_user, indent=2))

assert created_user["username"] == USERNAME
assert created_user["enabled"] is True
assert created_user["roles"] == []
print("user created without privileges.")

create response: {'created': True}
user snapshot:
{
  "username": "p13_analyst",
  "roles": [],
  "full_name": "Practical 13 Analyst",
  "email": "p13-analyst@example.test",
  "metadata": {},
  "enabled": true
}
user created without privileges.


## Task 3 - Create a role

The role grants cluster monitoring and read-only access to the
`p13-public-*` index pattern. It explicitly has no access to private indices.
The role is then assigned to the analyst user.

In [3]:
try:
    es.security.delete_role(name=ROLE)
except NotFoundError:
    pass

role_definition = {
    "cluster": ["monitor"],
    "indices": [
        {
            "names": ["p13-public-*"],
            "privileges": ["read", "view_index_metadata"],
        }
    ],
}
role_create_response = plain(es.security.put_role(name=ROLE, **role_definition))
es.security.put_user(
    username=USERNAME,
    password=USER_PASSWORD,
    roles=[ROLE],
    full_name="Practical 13 Analyst",
    email="p13-analyst@example.test",
)
role_snapshot = plain(es.security.get_role(name=ROLE))[ROLE]
assigned_user = plain(es.security.get_user(username=USERNAME))[USERNAME]

print("create response:", role_create_response)
print("role snapshot:")
print(json.dumps(role_snapshot, indent=2))
print("assigned roles:", assigned_user["roles"])

assert ROLE in assigned_user["roles"]
assert role_snapshot["indices"][0]["names"] == ["p13-public-*"]
assert "read" in role_snapshot["indices"][0]["privileges"]
print("role created and assigned.")

create response: {'role': {'created': True}}
role snapshot:
{
  "cluster": [
    "monitor"
  ],
  "indices": [
    {
      "names": [
        "p13-public-*"
      ],
      "privileges": [
        "read",
        "view_index_metadata"
      ],
      "allow_restricted_indices": false
    }
  ],
  "applications": [],
  "run_as": [],
  "metadata": {},
  "transient_metadata": {
    "enabled": true
  }
}
assigned roles: ['p13_public_reader']
role created and assigned.


## Task 4 - Restrict index access

Two indices are created with the same explicit mapping:

- `p13-public-events`: readable by the analyst role
- `p13-private-events`: intentionally not covered by the role

The analyst can count public documents but receives HTTP 403 for the private
index.

In [4]:
for name in [PUBLIC_INDEX, PRIVATE_INDEX]:
    delete_index(name)
    es.indices.create(index=name, **EVENT_MAPPING)

public_load = bulk_load(PUBLIC_INDEX, build_public_events(count=PUBLIC_TOTAL), "event_id")
private_load = bulk_load(PRIVATE_INDEX, build_private_events(count=PRIVATE_TOTAL), "event_id")
for name in [PUBLIC_INDEX, PRIVATE_INDEX]:
    es.indices.put_settings(index=name, settings={"refresh_interval": "1s"})
    es.indices.refresh(index=name)

public_count = int(es.count(index=PUBLIC_INDEX, query={"match_all": {}})["count"])
private_count = int(es.count(index=PRIVATE_INDEX, query={"match_all": {}})["count"])
analyst_client = Elasticsearch(
    ES_URL,
    basic_auth=(USERNAME, USER_PASSWORD),
    request_timeout=60,
)
analyst_public_count = int(analyst_client.count(index=PUBLIC_INDEX)["count"])
private_status = None
private_error = None
try:
    analyst_client.count(index=PRIVATE_INDEX)
except AuthorizationException as exc:
    private_status = exc.status_code
    private_error = str(exc)[:300]

print("public load:", public_load)
print("private load:", private_load)
print("admin counts:", public_count, private_count)
print("analyst public count:", analyst_public_count)
print("analyst private status:", private_status)
print("analyst private error:", private_error)

assert public_count == PUBLIC_TOTAL
assert private_count == PRIVATE_TOTAL
assert analyst_public_count == PUBLIC_TOTAL
assert private_status == 403
print("index access restriction verified.")

public load: {'documents': 1000, 'errors': 0, 'seconds': 3.687, 'docs_per_second': 271.3}
private load: {'documents': 500, 'errors': 0, 'seconds': 0.672, 'docs_per_second': 744.5}
admin counts: 1000 500
analyst public count: 1000
analyst private status: 403
analyst private error: AuthorizationException(403, 'security_exception', 'action [indices:data/read/search] is unauthorized for user [p13_analyst] with effective roles [p13_public_reader] on indices [p13-private-events], this action is granted by the index privileges [read,all]')
index access restriction verified.


## Task 5 - Generate an API key

A time-limited API key is created with a role descriptor that grants read-only
access to `p13-public-*`. The encoded secret is used for two test requests but is
not written to the artifact or printed.

API keys are preferable to embedding a user password in an application. The
key can be revoked independently.

In [5]:
api_key_name = f"p13-public-reader-{int(time.time())}"
api_key_response = plain(
    es.security.create_api_key(
        name=api_key_name,
        expiration="1d",
        role_descriptors={
            "p13_api_public_reader": {
                "cluster": ["monitor"],
                "indices": [
                    {
                        "names": ["p13-public-*"],
                        "privileges": ["read", "view_index_metadata"],
                    }
                ],
            }
        },
    )
)
api_client = Elasticsearch(
    ES_URL,
    api_key=api_key_response["encoded"],
    request_timeout=60,
)
api_public_count = int(api_client.count(index=PUBLIC_INDEX)["count"])
api_private_status = None
try:
    api_client.count(index=PRIVATE_INDEX)
except AuthorizationException as exc:
    api_private_status = exc.status_code

api_key_report = {
    "name": api_key_response["name"],
    "id": api_key_response["id"],
    "expiration": api_key_response["expiration"],
    "public_count": api_public_count,
    "private_status": api_private_status,
    "encoded_secret_stored": False,
}
print(json.dumps(api_key_report, indent=2))
assert api_public_count == PUBLIC_TOTAL
assert api_private_status == 403
print("API key authentication and index restriction verified.")

{
  "name": "p13-public-reader-1795155648",
  "id": "haud96EBccIezWQdYTDi",
  "expiration": 1795242049070,
  "public_count": 1000,
  "private_status": 403,
  "encoded_secret_stored": false
}
API key authentication and index restriction verified.


## Task 6 - Inspect cluster health

Cluster health reports shard allocation, node count, pending tasks, and the
overall status. A single-node cluster with all replicas set to zero should be
green.

In [6]:
cluster_health = plain(es.cluster.health())
cat_nodes = plain(es.cat.nodes(format="json"))
print("cluster health:")
print(json.dumps(cluster_health, indent=2))
print("nodes:")
print(json.dumps(cat_nodes, indent=2))

assert cluster_health["number_of_nodes"] == 1
assert cluster_health["status"] == "green"
assert cluster_health["unassigned_shards"] == 0
print("cluster health is green with one node.")

cluster health:
{
  "cluster_name": "p13-secured-cluster",
  "status": "green",
  "timed_out": false,
  "number_of_nodes": 1,
  "number_of_data_nodes": 1,
  "active_primary_shards": 8,
  "active_shards": 8,
  "relocating_shards": 0,
  "initializing_shards": 0,
  "unassigned_shards": 0,
  "unassigned_primary_shards": 0,
  "delayed_unassigned_shards": 0,
  "number_of_pending_tasks": 0,
  "number_of_in_flight_fetch": 0,
  "task_max_waiting_in_queue_millis": 0,
  "active_shards_percent_as_number": 100.0
}
nodes:
[
  {
    "ip": "172.20.0.2",
    "heap.percent": "24",
    "ram.percent": "98",
    "cpu": "23",
    "load_1m": "3.52",
    "load_5m": "3.55",
    "load_15m": "3.22",
    "node.role": "cdfhilmrstw",
    "master": "*",
    "name": "p13-node"
  }
]
cluster health is green with one node.


## Task 7 - Inspect node statistics

The node stats response is large, so the notebook records the operationally
useful fields: OS memory/CPU, process file descriptors, index totals, JVM
memory, GC counts, and thread-pool queues.

In [7]:
all_node_stats = plain(es.nodes.stats())
node_id, node_stats = next(iter(all_node_stats["nodes"].items()))
os_stats = node_stats["os"]
process_stats = node_stats["process"]
indices_stats = node_stats["indices"]
jvm_stats = node_stats["jvm"]

node_report = {
    "node_id": node_id,
    "name": node_stats["name"],
    "roles": node_stats["roles"],
    "os": {
        "cpu_percent": os_stats["cpu"]["percent"],
        "mem_total_in_bytes": os_stats["mem"]["total_in_bytes"],
        "mem_free_in_bytes": os_stats["mem"]["free_in_bytes"],
        "mem_used_percent": os_stats["mem"]["used_percent"],
        "swap_used_in_bytes": os_stats["swap"]["used_in_bytes"],
    },
    "process": {
        "open_file_descriptors": process_stats["open_file_descriptors"],
        "max_file_descriptors": process_stats["max_file_descriptors"],
        "cpu_percent": process_stats["cpu"]["percent"],
    },
    "indices": {
        "documents": indices_stats["docs"]["count"],
        "store_bytes": indices_stats["store"]["size_in_bytes"],
        "search_query_total": indices_stats["search"]["query_total"],
        "indexing_index_total": indices_stats["indexing"]["index_total"],
    },
    "jvm": {
        "uptime_in_millis": jvm_stats["uptime_in_millis"],
        "heap_used_in_bytes": jvm_stats["mem"]["heap_used_in_bytes"],
        "heap_max_in_bytes": jvm_stats["mem"]["heap_max_in_bytes"],
        "heap_used_percent": jvm_stats["mem"]["heap_used_percent"],
        "non_heap_used_in_bytes": jvm_stats["mem"]["non_heap_used_in_bytes"],
    },
    "thread_pool": {
        name: {
            "active": values["active"],
            "queue": values["queue"],
            "rejected": values["rejected"],
        }
        for name, values in node_stats["thread_pool"].items()
        if name in {"search", "write", "get", "bulk"}
    },
}
print(json.dumps(node_report, indent=2))
assert node_report["indices"]["documents"] >= PUBLIC_TOTAL + PRIVATE_TOTAL
print("node statistics collected.")

{
  "node_id": "S1plS8ZeRLu_6h52Xacocg",
  "name": "p13-node",
  "roles": [
    "data",
    "data_cold",
    "data_content",
    "data_frozen",
    "data_hot",
    "data_warm",
    "ingest",
    "master",
    "ml",
    "remote_cluster_client",
    "transform"
  ],
  "os": {
    "cpu_percent": 34,
    "mem_total_in_bytes": 4042948608,
    "mem_free_in_bytes": 68845568,
    "mem_used_percent": 98,
    "swap_used_in_bytes": 238858240
  },
  "process": {
    "open_file_descriptors": 629,
    "max_file_descriptors": 1048576,
    "cpu_percent": 18
  },
  "indices": {
    "documents": 2543,
    "store_bytes": 374479,
    "search_query_total": 24,
    "indexing_index_total": 3063
  },
  "jvm": {
    "uptime_in_millis": 1578446,
    "heap_used_in_bytes": 130617600,
    "heap_max_in_bytes": 536870912,
    "heap_used_percent": 24,
    "non_heap_used_in_bytes": 238065312
  },
  "thread_pool": {
    "get": {
      "active": 0,
      "queue": 0,
      "rejected": 0
    },
    "search": {
      "acti

## Task 8 - Investigate JVM memory usage

JVM heap is limited to `512 MB` by `ES_JAVA_OPTS`. The heap used percentage,
old-generation use, and GC counts show whether the node is under memory
pressure. Elasticsearch reserves headroom for segment merging, aggregations, and
the operating system filesystem cache.

In [8]:
jvm_mem = jvm_stats["mem"]
heap_used = jvm_mem["heap_used_in_bytes"]
heap_max = jvm_mem["heap_max_in_bytes"]
heap_used_percent = round(heap_used / heap_max * 100, 2)
pools = {
    name: {
        "used_in_bytes": values["used_in_bytes"],
        "max_in_bytes": values["max_in_bytes"],
        "peak_used_in_bytes": values["peak_used_in_bytes"],
    }
    for name, values in jvm_mem["pools"].items()
}
gc_collectors = {
    name: {
        "collection_count": values["collection_count"],
        "collection_time_in_millis": values["collection_time_in_millis"],
    }
    for name, values in jvm_stats["gc"]["collectors"].items()
}
jvm_report = {
    "heap_used_in_bytes": heap_used,
    "heap_max_in_bytes": heap_max,
    "heap_used_percent": heap_used_percent,
    "non_heap_used_in_bytes": jvm_mem["non_heap_used_in_bytes"],
    "pools": pools,
    "gc_collectors": gc_collectors,
    "uptime_in_millis": jvm_stats["uptime_in_millis"],
}
print(json.dumps(jvm_report, indent=2))
assert 0 <= heap_used_percent <= 100
assert heap_used <= heap_max
print("JVM memory is below its configured maximum.")

{
  "heap_used_in_bytes": 130617600,
  "heap_max_in_bytes": 536870912,
  "heap_used_percent": 24.33,
  "non_heap_used_in_bytes": 238065312,
  "pools": {
    "CodeHeap 'non-nmethods'": {
      "used_in_bytes": 2799872,
      "max_in_bytes": 5836800,
      "peak_used_in_bytes": 2859136
    },
    "Metaspace": {
      "used_in_bytes": 173331008,
      "max_in_bytes": 0,
      "peak_used_in_bytes": 173331008
    },
    "CodeHeap 'profiled nmethods'": {
      "used_in_bytes": 28805120,
      "max_in_bytes": 122908672,
      "peak_used_in_bytes": 28805120
    },
    "Compressed Class Space": {
      "used_in_bytes": 21673056,
      "max_in_bytes": 1073741824,
      "peak_used_in_bytes": 21673056
    },
    "young": {
      "used_in_bytes": 12582912,
      "max_in_bytes": 0,
      "peak_used_in_bytes": 310378496
    },
    "old": {
      "used_in_bytes": 115265848,
      "max_in_bytes": 536870912,
      "peak_used_in_bytes": 115265848
    },
    "survivor": {
      "used_in_bytes": 2768840,
 

## Task 9 - Configure slow logs

Search and indexing slow logs are configured with zero-millisecond thresholds so
this lab produces evidence immediately. In production, use meaningful
thresholds such as `1s` for search queries and `500ms` for indexing.

The container logs JSON slow-log entries to stdout, so the notebook reads them
with `docker logs`. The temporary `p13-slowlog-demo` index isolates the
indexing-slowlog example.

In [9]:
es.indices.put_settings(index=PUBLIC_INDEX, settings=SLOWLOG_SETTINGS)
public_slowlog_settings = plain(
    es.indices.get_settings(index=PUBLIC_INDEX, flat_settings=True)
)[PUBLIC_INDEX]["settings"]

for _ in range(3):
    es.search(index=PUBLIC_INDEX, query={"match": {"message": "error"}}, size=5)

delete_index(SLOWLOG_INDEX)
es.indices.create(index=SLOWLOG_INDEX, **EVENT_MAPPING)
es.indices.put_settings(index=SLOWLOG_INDEX, settings=SLOWLOG_SETTINGS)
for sequence in range(5):
    es.index(
        index=SLOWLOG_INDEX,
        id=f"slowlog-{sequence}",
        document={
            "event_id": f"SLOW-{sequence:03d}",
            "created_at": "2024-01-01T00:00:00Z",
            "service": "slowlog",
            "level": "info",
            "amount": 100 + sequence,
            "message": f"slow log demo event {sequence}",
            "visibility": "public",
        },
    )
es.indices.refresh(index=SLOWLOG_INDEX)
time.sleep(2)

all_slowlog_entries, slowlog_error = docker_slowlog_entries(minutes=5)
search_log_entries = [
    entry
    for entry in all_slowlog_entries
    if entry.get("event.dataset") == "elasticsearch.index_search_slowlog"
    and PUBLIC_INDEX in json.dumps(entry)
]
indexing_log_entries = [
    entry
    for entry in all_slowlog_entries
    if entry.get("event.dataset") == "elasticsearch.index_indexing_slowlog"
    and SLOWLOG_INDEX in json.dumps(entry)
]
slowlog_report = {
    "settings": {
        key: public_slowlog_settings.get(key)
        for key in SLOWLOG_SETTINGS
    },
    "docker_available": shutil.which("docker") is not None,
    "docker_error": slowlog_error,
    "search_entries": [slowlog_summary(entry) for entry in search_log_entries[-3:]],
    "indexing_entries": [slowlog_summary(entry) for entry in indexing_log_entries[-3:]],
}

print("slow log settings:")
print(json.dumps(slowlog_report["settings"], indent=2))
assert all(slowlog_report["settings"].values()), slowlog_report["settings"]
print("search slow-log entries:", len(search_log_entries))
print("indexing slow-log entries:", len(indexing_log_entries))
print(json.dumps(slowlog_report, indent=2))

if slowlog_report["docker_available"]:
    assert search_log_entries
    assert indexing_log_entries
delete_index(SLOWLOG_INDEX)
print("slow-log evidence collected.")

slow log settings:
{
  "index.search.slowlog.threshold.query.warn": "0ms",
  "index.search.slowlog.threshold.query.info": "0ms",
  "index.search.slowlog.threshold.fetch.warn": "0ms",
  "index.search.slowlog.threshold.fetch.info": "0ms",
  "index.indexing.slowlog.threshold.index.warn": "0ms",
  "index.indexing.slowlog.threshold.index.info": "0ms"
}
search slow-log entries: 6
indexing slow-log entries: 5
{
  "settings": {
    "index.search.slowlog.threshold.query.warn": "0ms",
    "index.search.slowlog.threshold.query.info": "0ms",
    "index.search.slowlog.threshold.fetch.warn": "0ms",
    "index.search.slowlog.threshold.fetch.info": "0ms",
    "index.indexing.slowlog.threshold.index.warn": "0ms",
    "index.indexing.slowlog.threshold.index.info": "0ms"
  },
  "docker_available": true,
  "docker_error": null,
  "search_entries": [
    {
      "level": "WARN",
      "logger": "index.search.slowlog.fetch",
      "message": "[p13-public-events][0]",
      "took": "11.6ms",
      "took_mill

slow-log evidence collected.


## Task 10 - Create a snapshot repository

The repository is a shared filesystem repository mounted into the container at
`/usr/share/elasticsearch/snapshots`. The host directory is
`Practical_13/snapshots`.

In [10]:
try:
    es.snapshot.delete_repository(name=SNAPSHOT_REPO)
except NotFoundError:
    pass

repository_response = plain(
    es.snapshot.create_repository(
        name=SNAPSHOT_REPO,
        repository={
            "type": "fs",
            "settings": {
                "location": "/usr/share/elasticsearch/snapshots",
                "compress": True,
            },
        },
    )
)
repository_snapshot = plain(es.snapshot.get_repository(name=SNAPSHOT_REPO))[SNAPSHOT_REPO]
print("create response:", repository_response)
print("repository:")
print(json.dumps(repository_snapshot, indent=2))
assert repository_snapshot["type"] == "fs"
print("snapshot repository registered.")

create response: {'acknowledged': True}
repository:
{
  "type": "fs",
  "uuid": "m2wJJ9RPSIq6kknPEE1HEQ",
  "settings": {
    "compress": "true",
    "location": "/usr/share/elasticsearch/snapshots"
  }
}
snapshot repository registered.


## Task 11 - Create a snapshot

Only `p13-public-events` is included and global cluster state is excluded, so
the snapshot contains the application data without security metadata or other
system indices.

In [11]:
try:
    es.snapshot.get(repository=SNAPSHOT_REPO, snapshot=SNAPSHOT_NAME)
    es.snapshot.delete(repository=SNAPSHOT_REPO, snapshot=SNAPSHOT_NAME)
except NotFoundError:
    pass

snapshot_started = time.perf_counter()
snapshot_response = plain(
    es.snapshot.create(
        repository=SNAPSHOT_REPO,
        snapshot=SNAPSHOT_NAME,
        body={
            "indices": PUBLIC_INDEX,
            "include_global_state": False,
        },
        wait_for_completion=True,
    )
)
snapshot_seconds = time.perf_counter() - snapshot_started
snapshot_info = plain(
    es.snapshot.get(repository=SNAPSHOT_REPO, snapshot=SNAPSHOT_NAME)
)["snapshots"][0]

snapshot_report = {
    "repository": SNAPSHOT_REPO,
    "snapshot": SNAPSHOT_NAME,
    "elapsed_seconds": round(snapshot_seconds, 3),
    "state": snapshot_info["state"],
    "indices": snapshot_info["indices"],
    "shards": snapshot_info["shards"],
    "include_global_state": snapshot_info["include_global_state"],
    "failures": snapshot_info["failures"],
}
print("create response:")
print(json.dumps(snapshot_response, indent=2))
print("snapshot:")
print(json.dumps(snapshot_report, indent=2))

assert snapshot_info["state"] == "SUCCESS"
assert snapshot_info["shards"]["failed"] == 0
assert PUBLIC_INDEX in snapshot_info["indices"]
print("snapshot completed successfully.")

create response:
{
  "snapshot": {
    "snapshot": "p13-public-snapshot",
    "uuid": "3487-LKBT8ukqYl9HFya2A",
    "repository": "p13-snapshot-repo",
    "version_id": 9039001,
    "version": "9.2.0",
    "indices": [
      "p13-public-events"
    ],
    "data_streams": [],
    "include_global_state": false,
    "state": "SUCCESS",
    "start_time": "2026-11-20T06:21:29.759Z",
    "start_time_in_millis": 1795155689759,
    "end_time": "2026-11-20T06:21:31.165Z",
    "end_time_in_millis": 1795155691165,
    "duration_in_millis": 1406,
    "failures": [],
    "shards": {
      "total": 1,
      "failed": 0,
      "successful": 1
    },
    "feature_states": []
  }
}
snapshot:
{
  "repository": "p13-snapshot-repo",
  "snapshot": "p13-public-snapshot",
  "elapsed_seconds": 3.232,
  "state": "SUCCESS",
  "indices": [
    "p13-public-events"
  ],
  "shards": {
    "total": 1,
    "failed": 0,
    "successful": 1
  },
  "include_global_state": false,
  "failures": []
}
snapshot completed suc

## Task 12 - Restore a snapshot

The snapshot is restored into a new index name with a rename pattern. Restoring
to a new name avoids deleting or overwriting the live source index.

In [12]:
delete_index(RESTORED_INDEX)
restore_started = time.perf_counter()
restore_response = plain(
    es.snapshot.restore(
        repository=SNAPSHOT_REPO,
        snapshot=SNAPSHOT_NAME,
        body={
            "indices": PUBLIC_INDEX,
            "rename_pattern": "p13-public-events",
            "rename_replacement": RESTORED_INDEX,
            "include_global_state": False,
            "ignore_unavailable": True,
        },
        wait_for_completion=True,
    )
)
restore_seconds = time.perf_counter() - restore_started
es.indices.refresh(index=RESTORED_INDEX)
restored_count = int(es.count(index=RESTORED_INDEX, query={"match_all": {}})["count"])
restored_mapping = plain(es.indices.get_mapping(index=RESTORED_INDEX))[RESTORED_INDEX]["mappings"]
restore_report = {
    "source": PUBLIC_INDEX,
    "destination": RESTORED_INDEX,
    "elapsed_seconds": round(restore_seconds, 3),
    "response": restore_response,
    "documents": restored_count,
    "source_documents": public_count,
    "mapping_dynamic": restored_mapping.get("dynamic"),
}
print(json.dumps(restore_report, indent=2))

assert restored_count == public_count
assert restored_mapping.get("dynamic") == "strict"
print("snapshot restored into a new index with matching document count.")

{
  "source": "p13-public-events",
  "destination": "p13-public-events-restored",
  "elapsed_seconds": 4.817,
  "response": {
    "snapshot": {
      "snapshot": "p13-public-snapshot",
      "indices": [
        "p13-public-events-restored"
      ],
      "shards": {
        "total": 1,
        "failed": 0,
        "successful": 1
      }
    }
  },
  "documents": 1000,
  "source_documents": 1000,
  "mapping_dynamic": "strict"
}
snapshot restored into a new index with matching document count.


## Task 13 - Investigate an unassigned shard

A single-node cluster cannot allocate a replica to a second node. Creating an
index with one replica therefore leaves the replica unassigned and turns cluster
health yellow. `_cluster/allocation/explain` identifies `same_shard` as the
blocking decider. Reducing the replica count to zero resolves the condition.

In [13]:
delete_index(UNASSIGNED_INDEX)
es.indices.create(
    index=UNASSIGNED_INDEX,
    settings={"number_of_shards": 1, "number_of_replicas": 1},
    mappings=EVENT_MAPPING["mappings"],
)
es.index(
    index=UNASSIGNED_INDEX,
    id="unassigned-1",
    document={
        "event_id": "UNASSIGNED-001",
        "created_at": "2024-01-01T00:00:00Z",
        "service": "allocation",
        "level": "warn",
        "amount": 1,
        "message": "unassigned shard demo",
        "visibility": "public",
    },
    refresh=True,
)
health_with_unassigned = plain(es.cluster.health(index=UNASSIGNED_INDEX))
unassigned_shards = [
    row
    for row in plain(es.cat.shards(index=UNASSIGNED_INDEX, format="json"))
    if row["state"] == "UNASSIGNED"
]
allocation_explain = plain(
    es.cluster.allocation_explain(
        index=UNASSIGNED_INDEX,
        shard=0,
        primary=False,
    )
)
blocking_deciders = [
    decider
    for decision in allocation_explain["node_allocation_decisions"]
    for decider in decision["deciders"]
]

print("health with replica:")
print(json.dumps(health_with_unassigned, indent=2))
print("unassigned shard rows:")
print(json.dumps(unassigned_shards, indent=2))
print("can allocate:", allocation_explain["can_allocate"])
print("blocking deciders:")
print(json.dumps(blocking_deciders, indent=2))

assert health_with_unassigned["status"] == "yellow"
assert unassigned_shards
assert allocation_explain["can_allocate"] == "no"
assert any(decider["decider"] == "same_shard" for decider in blocking_deciders)

es.indices.put_settings(
    index=UNASSIGNED_INDEX,
    settings={"index.number_of_replicas": 0},
)
health_after_fix = plain(
    es.cluster.health(
        index=UNASSIGNED_INDEX,
        wait_for_status="green",
        timeout="30s",
    )
)
shards_after_fix = plain(es.cat.shards(index=UNASSIGNED_INDEX, format="json"))
unassigned_report = {
    "index": UNASSIGNED_INDEX,
    "health_before": health_with_unassigned,
    "unassigned_rows_before": unassigned_shards,
    "allocation_explain": allocation_explain,
    "blocking_deciders": blocking_deciders,
    "health_after": health_after_fix,
    "shards_after": shards_after_fix,
}
print("health after setting replicas to zero:")
print(json.dumps(health_after_fix, indent=2))
assert health_after_fix["status"] == "green"
assert not [
    row for row in shards_after_fix if row["state"] == "UNASSIGNED"
]
print("unassigned replica investigated and resolved.")

health with replica:
{
  "cluster_name": "p13-secured-cluster",
  "status": "yellow",
  "timed_out": false,
  "number_of_nodes": 1,
  "number_of_data_nodes": 1,
  "active_primary_shards": 1,
  "active_shards": 1,
  "relocating_shards": 0,
  "initializing_shards": 0,
  "unassigned_shards": 1,
  "unassigned_primary_shards": 0,
  "delayed_unassigned_shards": 0,
  "number_of_pending_tasks": 0,
  "number_of_in_flight_fetch": 0,
  "task_max_waiting_in_queue_millis": 0,
  "active_shards_percent_as_number": 50.0
}
unassigned shard rows:
[
  {
    "index": "p13-unassigned-demo",
    "shard": "0",
    "prirep": "r",
    "state": "UNASSIGNED",
    "docs": null,
    "store": null,
    "dataset": null,
    "ip": null,
    "node": null
  }
]
can allocate: no
blocking deciders:
[
  {
    "decider": "same_shard",
    "decision": "NO",
    "explanation": "a copy of this shard is already allocated to this node [[p13-unassigned-demo][0], node[S1plS8ZeRLu_6h52Xacocg], [P], s[STARTED], a[id=xR3MfuQhRA-MopB

health after setting replicas to zero:
{
  "cluster_name": "p13-secured-cluster",
  "status": "green",
  "timed_out": false,
  "number_of_nodes": 1,
  "number_of_data_nodes": 1,
  "active_primary_shards": 1,
  "active_shards": 1,
  "relocating_shards": 0,
  "initializing_shards": 0,
  "unassigned_shards": 0,
  "unassigned_primary_shards": 0,
  "delayed_unassigned_shards": 0,
  "number_of_pending_tasks": 0,
  "number_of_in_flight_fetch": 0,
  "task_max_waiting_in_queue_millis": 0,
  "active_shards_percent_as_number": 100.0
}
unassigned replica investigated and resolved.


## Task 14 - Investigate an indexing failure

Two controlled failures are tested against a strict-mapping index:

1. A string is sent to an `integer` field and fails with
   `document_parsing_exception`.
2. An unknown field is sent under `dynamic: strict` and fails with
   `strict_dynamic_mapping_exception`.

Both requests return HTTP 400, and neither document is added. A corrected
document is then indexed successfully.

In [14]:
delete_index(FAILURE_INDEX)
es.indices.create(index=FAILURE_INDEX, **FAILURE_MAPPING)
es.index(
    index=FAILURE_INDEX,
    id="ok-1",
    document={"event_id": "OK-1", "amount": 100},
    refresh=True,
)

failure_cases = []
try:
    es.index(
        index=FAILURE_INDEX,
        id="bad-type",
        document={"event_id": "BAD-TYPE", "amount": "not-a-number"},
    )
except BadRequestError as exc:
    failure_cases.append(
        {
            "case": "integer field receives a non-numeric string",
            "status": exc.status_code,
            "body": plain(exc.body),
        }
    )

try:
    es.index(
        index=FAILURE_INDEX,
        id="bad-field",
        document={"event_id": "BAD-FIELD", "amount": 200, "unknown_field": "x"},
    )
except BadRequestError as exc:
    failure_cases.append(
        {
            "case": "unknown field under dynamic strict",
            "status": exc.status_code,
            "body": plain(exc.body),
        }
    )

count_after_failures = int(es.count(index=FAILURE_INDEX)["count"])
es.index(
    index=FAILURE_INDEX,
    id="ok-2",
    document={"event_id": "OK-2", "amount": 200},
    refresh=True,
)
count_after_fix = int(es.count(index=FAILURE_INDEX)["count"])

failure_report = {
    "index": FAILURE_INDEX,
    "cases": failure_cases,
    "count_after_failures": count_after_failures,
    "count_after_corrected_document": count_after_fix,
    "resolution": "Correct the amount value or add an explicit mapping for the new field.",
}
print(json.dumps(failure_report, indent=2))

assert len(failure_cases) == 2
assert all(case["status"] == 400 for case in failure_cases)
assert count_after_failures == 1
assert count_after_fix == 2
print("indexing failures captured and corrected.")

{
  "index": "p13-failure-demo",
  "cases": [
    {
      "case": "integer field receives a non-numeric string",
      "status": 400,
      "body": {
        "error": {
          "root_cause": [
            {
              "type": "document_parsing_exception",
              "reason": "[1:33] failed to parse field [amount] of type [integer] in document with id 'bad-type'. Preview of field's value: 'not-a-number'"
            }
          ],
          "type": "document_parsing_exception",
          "reason": "[1:33] failed to parse field [amount] of type [integer] in document with id 'bad-type'. Preview of field's value: 'not-a-number'",
          "caused_by": {
            "type": "number_format_exception",
            "reason": "For input string: \"not-a-number\""
          }
        },
        "status": 400
      }
    },
    {
      "case": "unknown field under dynamic strict",
      "status": 400,
      "body": {
        "error": {
          "root_cause": [
            {
            

## Write artifacts

Two JSON reports are written: security/monitoring data and snapshot/failure
analysis. The API key secret is intentionally omitted.

In [15]:
security_report = {
    "environment": security_environment,
    "user": created_user,
    "role": role_snapshot,
    "index_access": {
        "public_index": PUBLIC_INDEX,
        "private_index": PRIVATE_INDEX,
        "public_documents": public_count,
        "private_documents": private_count,
        "analyst_public_count": analyst_public_count,
        "analyst_private_status": private_status,
        "analyst_private_error": private_error,
    },
    "api_key": api_key_report,
    "cluster_health": cluster_health,
    "node": node_report,
    "jvm": jvm_report,
    "slow_log": slowlog_report,
}

operations_report = {
    "snapshot_repository": repository_snapshot,
    "snapshot": snapshot_report,
    "restore": restore_report,
    "unassigned_shard": unassigned_report,
    "indexing_failure": failure_report,
}

(ARTIFACTS / "security_report.json").write_text(
    json.dumps(plain(security_report), indent=2), encoding="utf-8"
)
(ARTIFACTS / "operations_report.json").write_text(
    json.dumps(plain(operations_report), indent=2), encoding="utf-8"
)

print("wrote:", ARTIFACTS / "security_report.json")
print("wrote:", ARTIFACTS / "operations_report.json")
print("public documents:", public_count)
print("private documents:", private_count)
print("cluster health:", cluster_health["status"])
print("snapshot state:", snapshot_report["state"])
print("restored documents:", restored_count)
print("slow-log search entries:", len(slowlog_report["search_entries"]))
print("slow-log indexing entries:", len(slowlog_report["indexing_entries"]))

assert (ARTIFACTS / "security_report.json").is_file()
assert (ARTIFACTS / "operations_report.json").is_file()
assert "encoded" not in security_report["api_key"]
print("artifact assertions passed.")

wrote: C:\D-Drive\Elastic Search Practicals\Practical_13\artifacts\security_report.json
wrote: C:\D-Drive\Elastic Search Practicals\Practical_13\artifacts\operations_report.json
public documents: 1000
private documents: 500
cluster health: green
snapshot state: SUCCESS
restored documents: 1000
slow-log search entries: 3
slow-log indexing entries: 3
artifact assertions passed.
